# Workflow: branching and parallel steps

This is notebook 2 of 3 in the workflow series. It builds on
[Notebook 1](1_workflow_intro.ipynb), which introduced linear workflows.

By default the workflow transform links each step to the one before it in the
list. Real workflows often branch: two tests run in parallel, or results from
two steps are combined before a final decision.

This notebook shows how to use `preceded_by` and `step_id` to express any
directed acyclic graph (DAG) of steps.

```
Step 1: Raw Material Inspection
        |
        +---> Step 2: Tensile Test --+
        |                           |
        +---> Step 3: Hardness Test -+--> Step 4: Quality Report
```

- Steps 2 and 3 both follow Step 1 (fan-out).
- Step 4 follows both Step 2 and Step 3 (fan-in).


In [1]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib, rdflib
from semantic_schemas import Schema

HERE     = pathlib.Path().resolve()   # schemas/workflow/PMDCo/docs/
WORKFLOW = HERE.parent                # schemas/workflow/PMDCo/

schema = Schema(WORKFLOW)

print("Python :", sys.executable)

Python : /root/semantic-dataspace/.venv/bin/python3


## Expressing a branching workflow

To control ordering explicitly, give each step a short `step_id`.
Then use `preceded_by` to list the IDs of all steps that must finish first.

| Technique | How |
|---|---|
| Fan-out (one step spawns two) | Give two steps the same `preceded_by` |
| Fan-in (two steps merge into one) | List both predecessors in `preceded_by` |
| Sequential (default) | Omit `preceded_by`; the transform auto-links in list order |


In [3]:
workflow_input = {
    'workflow_name': 'Parallel QA workflow, batch A',
    'description': 'QA workflow with two parallel tests followed by a combined report.',
    'steps': [
        {
            'step_id': 'step-inspection',
            'label':   'Raw Material Inspection',
            'description': 'Visual and dimensional check of the incoming steel sheet.',
            # No preceded_by: this is the first step.
        },
        {
            'step_id':     'step-tensile',
            'label':       'Tensile Test',
            'description': 'Room-temperature tensile test, ISO 6892-1.',
            'preceded_by': ['step-inspection'],   # follows inspection
        },
        {
            'step_id':     'step-hardness',
            'label':       'Hardness Test',
            'description': 'Vickers hardness measurement as parallel QA check.',
            'preceded_by': ['step-inspection'],   # also follows inspection (fan-out)
        },
        {
            'step_id':     'step-report',
            'label':       'Quality Report',
            'description': 'Combine tensile and hardness results into a pass/fail decision.',
            'preceded_by': ['step-tensile', 'step-hardness'],   # fan-in from both
        },
    ],
}

oold_doc = schema.transform(workflow_input)

print('Step order (from OO-LD):')
for s in oold_doc['has_part']:
    pre = s.get('preceded_by', ['(first)'])
    print(f"  {s['id']}")
    print(f"    preceded_by: {pre}")

Step order (from OO-LD):
  step-inspection
    preceded_by: ['(first)']
  step-tensile
    preceded_by: ['step-inspection']
  step-hardness
    preceded_by: ['step-inspection']
  step-report
    preceded_by: ['step-tensile', 'step-hardness']


## Build the RDF graph and validate


In [4]:
flat = schema.to_graph(workflow_input)

print(f"Graph contains {len(flat)} triples.")

conforms, violations = schema.validate(flat)
print(f"SHACL conforms: {conforms}")

Graph contains 24 triples.
SHACL conforms: True


## Query: who precedes whom?

The SPARQL query below retrieves all predecessor relationships from the graph,
making the branch structure explicit.


In [5]:
SPARQL = """
PREFIX bfo:     <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?step_label ?preceded_by_label
WHERE {
  ?step rdfs:label ?step_label .
  FILTER NOT EXISTS { ?step dcterms:conformsTo ?_ . }  # exclude the workflow root
  OPTIONAL {
    ?step bfo:0000062 ?prev .
    ?prev rdfs:label ?preceded_by_label .
  }
}
ORDER BY ?step_label
"""

rows = list(flat.query(SPARQL))
print('{:<30}  {}'.format('Step', 'Preceded by'))
print('-' * 60)
for r in rows:
    pre = str(r.preceded_by_label) if r.preceded_by_label else '(first)'
    print(f'{str(r.step_label):<30}  {pre}')

print()
print('Note: Quality Report appears twice because it has two predecessors.')

Step                            Preceded by
------------------------------------------------------------
Hardness Test                   Raw Material Inspection
Quality Report                  Tensile Test
Quality Report                  Hardness Test
Raw Material Inspection         (first)
Tensile Test                    Raw Material Inspection

Note: Quality Report appears twice because it has two predecessors.


## Summary

| Pattern | How to express it |
|---|---|
| Linear (default) | List steps in order; no `preceded_by` needed |
| Fan-out | Give two steps the same `preceded_by` entry |
| Fan-in | List multiple predecessors in a single `preceded_by` |
| Any DAG | Combine fan-out and fan-in as needed |

`step_id` lets you assign predictable IDs so that `preceded_by` references
remain readable and do not depend on auto-generated slugs.

---

**Next:** [Notebook 3](3_workflow_cross_domain.ipynb) applies these concepts
to a full cross-domain workflow spanning material production, tensile testing,
constitutive model calibration, and FEM material card export.
